# Project Milestone Two: Modeling and Feature Engineering

### Overview

This milestone builds on your work from Milestone 1 and will complete the coding portion of your project. You will:

1. Pick 3 modeling algorithms from those we have studied.
2. Evaluate baseline models using default settings.
3. Engineer new features and re-evaluate models.
4. Use feature selection techniques and re-evaluate.
5. Fine-tune for optimal performance.
6. Select your best model and report on your results. 

You must do all work in this notebook and upload to your team leader's account in Gradescope. There is no
Individual Assessment for this Milestone. 


In [55]:
# ===================================
# Useful Imports: Add more as needed
# ===================================

# Standard Libraries
import os
import time
import math
import io
import zipfile
import requests
from urllib.parse import urlparse
from itertools import chain, combinations

# Data Science Libraries
import numpy as np
import pandas as pd
import seaborn as sns

# Visualization
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import matplotlib.ticker as mticker  # Optional: Format y-axis labels as dollars
import seaborn as sns

# Scikit-learn (Machine Learning)
from sklearn.model_selection import (
    train_test_split, 
    cross_val_score, 
    GridSearchCV, 
    RandomizedSearchCV, 
    RepeatedKFold
)
from sklearn.preprocessing import StandardScaler, OrdinalEncoder
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_squared_error
from sklearn.feature_selection import SequentialFeatureSelector, f_regression, SelectKBest
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.ensemble import BaggingRegressor, RandomForestRegressor, GradientBoostingRegressor

# Progress Tracking

from tqdm import tqdm

# =============================
# Global Variables
# =============================
random_state = 42

# =============================
# Utility Functions
# =============================

# Format y-axis labels as dollars with commas (optional)
def dollar_format(x, pos):
    return f'${x:,.0f}'

# Convert seconds to HH:MM:SS format
def format_hms(seconds):
    return time.strftime("%H:%M:%S", time.gmtime(seconds))



### Prelude: Load your Preprocessed Dataset from Milestone 1

In Milestone 1, you handled missing values, encoded categorical features, and explored your data. Before you begin this milestone, you’ll need to load that cleaned dataset and prepare it for modeling. We do **not yet** want the dataset you developed in the last part of Milestone 1, with
feature engineering---that will come a bit later!

Here’s what to do:

1. Return to your Milestone 1 notebook and rerun your code through Part 3, where your dataset was fully cleaned (assume it’s called `df_cleaned`).

2. **Save** the cleaned dataset to a file by running:

>   df_cleaned.to_csv("zillow_cleaned.csv", index=False)

3. Switch to this notebook and **load** the saved data:

>   df = pd.read_csv("zillow_cleaned.csv")

4. Create a **train/test split** using `train_test_split`.  
   
6. **Standardize** the features (but not the target!) using **only the training data.** This ensures consistency across models without introducing data leakage from the test set:

>   scaler = StandardScaler()   
>   X_train_scaled = scaler.fit_transform(X_train)    
  
**Notes:** 

- You will have to redo the scaling step if you introduce new features (which have to be scaled as well).


In [84]:
df_cleaned = pd.read_csv('zillow_cleaned.csv')
#test train split 
y = df_cleaned['taxvaluedollarcnt']
X = df_cleaned.drop('taxvaluedollarcnt', axis=1)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


In [86]:
X_train.head()

,Unnamed: 0,bedroomcnt,bathroomcnt,roomcnt,numberofstories,yearbuilt,calculatedfinishedsquarefeet,lotsizesquarefeet,garagecarcnt,latitude,longitude,regionidzip,poolcnt,prop_age
49046,50075,3.0,3.0,0.0,1.0,1966.0,2614.0,24741.0,0.0,34149214.0,-118622654.0,96387.0,1.0,51.0
72330,73849,2.0,1.0,0.0,1.0,1950.0,610.0,9655.0,0.0,34057699.0,-118052895.0,96480.0,0.0,67.0
56168,57363,5.0,4.0,9.0,1.0,1951.0,3886.0,14352.0,2.0,33770272.0,-117880409.0,97006.0,1.0,66.0
55902,57092,2.0,2.0,0.0,1.0,1950.0,1412.0,4934.0,0.0,33893654.0,-118084509.0,96193.0,0.0,67.0
70646,72117,3.0,2.0,6.0,1.0,1963.0,1527.0,6060.0,2.0,33800557.0,-118040735.0,96218.0,1.0,54.0


In [60]:
#standardize data 
scaler = StandardScaler()
x_train_scaled = scaler.fit_transform(X_train)
x_test_scaled = scaler.fit_transform(X_test)

### Part 1: Picking Three Models and Establishing Baselines [6 pts]

Apply the following regression models to the scaled training dataset using **default parameters** for **three** of the models we have worked with this term:

- Linear Regression
- Ridge Regression
- Lasso Regression
- Decision Tree Regression
- Bagging
- Random Forest
- Gradient Boosting Trees

For each of the three models:
- Use **repeated cross-validation** (e.g., 5 folds, 5 repeats).
- Report the **mean and standard deviation of CV MAE Score**. 


In [62]:
# Add as many cells as you need
# Linear Regression 
from sklearn.metrics import mean_absolute_error
model_linear_reg = LinearRegression()

cv_strat = RepeatedKFold(n_splits=5, n_repeats=5, random_state=42)

scoreLinear = cross_val_score(model_linear_reg, x_train_scaled, y_train, cv=cv_strat, scoring='neg_mean_absolute_error')
mae_scores = -scoreLinear


print(f"Mean MAE: {mae_scores.mean():.4f}")
print(f"Standard Deviation: {mae_scores.std():.4f}")

model_linear_reg.fit(x_train_scaled, y_train)

# ---- Predict on test data ----
y_pred_test = model_linear_reg.predict(x_test_scaled)

# ---- Evaluate test performance ----
test_mae = mean_absolute_error(y_test, y_pred_test)

print(f"Test MAE: {test_mae:.4f}")

Mean MAE: 195898.5243
Standard Deviation: 1629.0584
Test MAE: 196496.3278


In [63]:
#Lasso Regression
model_lasso = Lasso(alpha=0.1) 
cv_strat = RepeatedKFold(n_splits=5, n_repeats=5, random_state=42)

scoreLinear_lasso = cross_val_score(model_lasso, x_train_scaled, y_train, cv=cv_strat, scoring='neg_mean_absolute_error')
mae_scores_lasso = -scoreLinear_lasso


print(f"Mean MAE: {mae_scores_lasso.mean():.4f}")
print(f"Standard Deviation: {mae_scores_lasso.std():.4f}")

model_lasso.fit(x_train_scaled, y_train)

# ---- Predict on test data ----
y_pred_test = model_lasso.predict(x_test_scaled)

# ---- Evaluate test performance ----
test_mae = mean_absolute_error(y_test, y_pred_test)

print(f"Test MAE: {test_mae:.4f}")

/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/linear_model/_coordinate_descent.py:716: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 4.731e+13, tolerance: 6.923e+11
  model = cd_fast.enet_coordinate_descent(
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/linear_model/_coordinate_descent.py:716: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 4.478e+13, tolerance: 6.920e+11
  model = cd_fast.enet_coordinate_descent(
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/linear_model/_coordinate_descent.py:716: ConvergenceWarning: Objective did not converge. You might want to increase the number of iter

Mean MAE: 195898.4865
Standard Deviation: 1629.0580
Test MAE: 196496.2927


/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/linear_model/_coordinate_descent.py:716: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 5.649e+13, tolerance: 8.636e+11
  model = cd_fast.enet_coordinate_descent(


In [65]:
#gradient boosting tree
model_gradbst = GradientBoostingRegressor()

# Cross-validation
cv_strat = RepeatedKFold(n_splits=5, n_repeats=5, random_state=42)

scoreLinear_grdtbst = cross_val_score(
    model_gradbst, 
    x_train_scaled, 
    y_train, 
    cv=cv_strat, 
    scoring='neg_mean_absolute_error'
)

mae_scores_grdbst = -scoreLinear_grdtbst

print(f"Mean MAE: {mae_scores_grdbst.mean():.4f}")
print(f"Standard Deviation: {mae_scores_grdbst.std():.4f}")

# ---- Fit on full training data ----
model_gradbst.fit(x_train_scaled, y_train)

# ---- Predict on test data ----
y_pred_test = model_gradbst.predict(x_test_scaled)

# ---- Evaluate test performance ----
test_mae = mean_absolute_error(y_test, y_pred_test)

print(f"Test MAE: {test_mae:.4f}")



Mean MAE: 171668.1446
Standard Deviation: 1730.5199
Test MAE: 200688.0543


### Part 1: Discussion [3 pts]

In a paragraph or well-organized set of bullet points, briefly compare and discuss:

  - Which model performed best overall?
  - Which was most stable (lowest std)?
  - Any signs of overfitting or underfitting?

Among the three models, the Gradient Boosting Regressor produced the lowest mean MAE, indicating the best predictive performance. This result is expected, as gradient boosting is capable of capturing nonlinear relationships and interactions between features that linear models cannot. Linear Regression and Lasso Regression performed similarly, with slightly higher MAE values, suggesting that a purely linear relationship may not fully capture the complexity of the data.

The relatively small standard deviations across cross-validation folds indicate that model performance is consistent and stable. Based on these baseline results, the Gradient Boosting model appears to be the strongest starting point for further improvement. In the next steps, feature engineering and model tuning will be explored to improve performance across all models.

### Part 2: Feature Engineering [6 pts]

Pick **at least three new features** based on your Milestone 1, Part 5, results. You may pick new ones or
use the same ones you chose for Milestone 1. 

Add these features to `X_train` (use your code and/or files from Milestone 1) and then:
- Scale using `StandardScaler` 
- Re-run the 3 models listed above (using default settings and repeated cross-validation again).
- Report the **mean and standard deviation of CV MAE Scores**.  


In [77]:
df_cleaned.head()

,Unnamed: 0,bedroomcnt,bathroomcnt,roomcnt,numberofstories,yearbuilt,calculatedfinishedsquarefeet,lotsizesquarefeet,garagecarcnt,latitude,longitude,regionidzip,poolcnt,taxvaluedollarcnt,prop_age,price_per_sqft
0,0,4.0,3.5,0.0,1.0,1998.0,3100.0,4506.0,2.0,33634931.0,-117869207.0,96978.0,0.0,1023282.0,19.0,330.090968
1,1,2.0,1.0,5.0,1.0,1967.0,1465.0,12647.0,1.0,34449266.0,-119281531.0,97099.0,0.0,464000.0,50.0,316.723549
2,2,3.0,2.0,6.0,1.0,1962.0,1243.0,8432.0,2.0,33886168.0,-117823170.0,97078.0,1.0,564778.0,55.0,454.366854
3,3,4.0,3.0,0.0,1.0,1970.0,2376.0,13038.0,0.0,34245180.0,-118240722.0,96330.0,1.0,145143.0,47.0,61.087121
4,4,3.0,3.0,0.0,1.0,1964.0,1312.0,278581.0,0.0,34185120.0,-118414640.0,96451.0,1.0,119407.0,53.0,91.011433


In [92]:
# Add as many cells as you need
# Price per square foot
df_cleaned['price_per_sqft'] = df_cleaned['taxvaluedollarcnt'] / df_cleaned['calculatedfinishedsquarefeet']

#average room size 
df_cleaned['avg_room_size'] = (
df_cleaned['calculatedfinishedsquarefeet'] / df_cleaned['roomcnt'])

# Age of home
df_cleaned['home_age'] = 2024 - df_cleaned['yearbuilt']

In [88]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

scaler = StandardScaler()

x_train_scaled = scaler.fit_transform(X_train)
x_test_scaled = scaler.transform(X_test)

In [89]:
model_linear_reg = LinearRegression()

cv_strat = RepeatedKFold(n_splits=5, n_repeats=5, random_state=42)

scoreLinear = cross_val_score(model_linear_reg, x_train_scaled, y_train, cv=cv_strat, scoring='neg_mean_absolute_error')
mae_scores = -scoreLinear


print(f"Mean MAE: {mae_scores.mean():.4f}")
print(f"Standard Deviation: {mae_scores.std():.4f}")

model_linear_reg.fit(x_train_scaled, y_train)

# ---- Predict on test data ----
y_pred_test = model_linear_reg.predict(x_test_scaled)

# ---- Evaluate test performance ----
test_mae = mean_absolute_error(y_test, y_pred_test)

print(f"Test MAE: {test_mae:.4f}")

Mean MAE: 195898.5243
Standard Deviation: 1629.0584
Test MAE: 197203.6556


In [90]:
#Lasso Regression
model_lasso = Lasso(alpha=0.1) 
cv_strat = RepeatedKFold(n_splits=5, n_repeats=5, random_state=42)

scoreLinear_lasso = cross_val_score(model_lasso, x_train_scaled, y_train, cv=cv_strat, scoring='neg_mean_absolute_error')
mae_scores_lasso = -scoreLinear_lasso


print(f"Mean MAE: {mae_scores_lasso.mean():.4f}")
print(f"Standard Deviation: {mae_scores_lasso.std():.4f}")
model_lasso.fit(x_train_scaled, y_train)

# ---- Predict on test data ----
y_pred_test = model_lasso.predict(x_test_scaled)

# ---- Evaluate test performance ----
test_mae = mean_absolute_error(y_test, y_pred_test)

print(f"Test MAE: {test_mae:.4f}")

/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/linear_model/_coordinate_descent.py:716: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 4.731e+13, tolerance: 6.923e+11
  model = cd_fast.enet_coordinate_descent(
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/linear_model/_coordinate_descent.py:716: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 4.478e+13, tolerance: 6.920e+11
  model = cd_fast.enet_coordinate_descent(
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/linear_model/_coordinate_descent.py:716: ConvergenceWarning: Objective did not converge. You might want to increase the number of iter

Mean MAE: 195898.4865
Standard Deviation: 1629.0580
Test MAE: 197203.6180


/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/linear_model/_coordinate_descent.py:716: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 5.649e+13, tolerance: 8.636e+11
  model = cd_fast.enet_coordinate_descent(


In [91]:
#gradient boosting tree
model_gradbst = GradientBoostingRegressor()
cv_strat = RepeatedKFold(n_splits=5, n_repeats=5, random_state=42)

scoreLinear_grdtbst = cross_val_score(model_gradbst, x_train_scaled, y_train, cv=cv_strat, scoring='neg_mean_absolute_error')
mae_scores_grdbst = -scoreLinear_grdtbst


print(f"Mean MAE: {mae_scores_grdbst.mean():.4f}")
print(f"Standard Deviation: {mae_scores_grdbst.std():.4f}")

# ---- Fit on full training data ----
model_gradbst.fit(x_train_scaled, y_train)

# ---- Predict on test data ----
y_pred_test = model_gradbst.predict(x_test_scaled)

# ---- Evaluate test performance ----
test_mae = mean_absolute_error(y_test, y_pred_test)

print(f"Test MAE: {test_mae:.4f}")

Mean MAE: 171667.7699
Standard Deviation: 1731.9446
Test MAE: 173018.8727


### Part 2: Discussion [3 pts]

Reflect on the impact of your new features:

- Did any models show notable improvement in performance?

- Which new features seemed to help — and in which models?

- Do you have any hypotheses about why a particular feature helped (or didn’t)?




Three new features were created to improve model performance: “I engineered new features including average room size and bathroom-to-bedroom ratio to better capture property layout and functionality. These features were designed to capture interactions and reduce skew in the data that may not be fully represented in the original variables.

Feature engineering had mixed effects across models. For linear and Lasso regression, performance remained nearly unchanged, with a slight increase in MAE, indicating that the new features did not add meaningful predictive value for these models. However, Gradient Boosting showed a substantial improvement, with test MAE decreasing significantly after feature engineering. This suggests that tree-based models were better able to leverage the nonlinear relationships introduced by the new features. Overall, feature engineering was beneficial, particularly for more complex models.”

### Part 3: Feature Selection [6 pts]

Using the full set of features (original + engineered):
- Apply **feature selection** methods to investigate whether you can improve performance.
  - You may use forward selection, backward selection, or feature importance from tree-based models.
- For each model, identify the **best-performing subset of features**.
- Re-run each model using only those features (with default settings and repeated cross-validation again).
- Report the **mean and standard deviation of CV MAE Scores**.  


In [5]:
# Add as many cells as you need


### Part 3: Discussion [3 pts]

Analyze the effect of feature selection on your models:

- Did performance improve for any models after reducing the number of features?

- Which features were consistently retained across models?

- Were any of your newly engineered features selected as important?


> Your text here

### Part 4: Fine-Tuning Your Three Models [6 pts]

In this final phase of Milestone 2, you’ll select and refine your **three most promising models and their corresponding data pipelines** based on everything you've done so far, and pick a winner!

1. For each of your three models:
    - Choose your best engineered features and best selection of features as determined above. 
   - Perform hyperparameter tuning using `sweep_parameters`, `GridSearchCV`, `RandomizedSearchCV`, `Optuna`, etc. as you have practiced in previous homeworks. 
3. Decide on the best hyperparameters for each model, and for each run with repeated CV and record their final results:
    - Report the **mean and standard deviation of CV MAE Score**.  

In [6]:
# Add as many cells as you need


### Part 4: Discussion [3 pts]

Reflect on your tuning process and final results:

- What was your tuning strategy for each model? Why did you choose those hyperparameters?
- Did you find that certain types of preprocessing or feature engineering worked better with specific models?


> Your text here

### Part 5: Final Model and Design Reassessment [6 pts]

In this part, you will finalize your best-performing model.  You’ll also consolidate and present the key code used to run your model on the preprocessed dataset.
**Requirements:**

- Decide one your final model among the three contestants. 

- Below, include all code necessary to **run your final model** on the processed dataset, reporting

    - Mean and standard deviation of CV MAE Score.
    
    - Test score on held-out test set. 




In [7]:
# Add as many cells as you need


### Part 5: Discussion [8 pts]

In this final step, your goal is to synthesize your entire modeling process and assess how your earlier decisions influenced the outcome. Please address the following:

1. Model Selection:
- Clearly state which model you selected as your final model and why.

- What metrics or observations led you to this decision?

- Were there trade-offs (e.g., interpretability vs. performance) that influenced your choice?

2. Revisiting an Early Decision

- Identify one specific preprocessing or feature engineering decision from Milestone 1 (e.g., how you handled missing values, how you scaled or encoded a variable, or whether you created interaction or polynomial terms).

- Explain the rationale for that decision at the time: What were you hoping it would achieve?

- Now that you've seen the full modeling pipeline and final results, reflect on whether this step helped or hindered performance. Did you keep it, modify it, or remove it?

- Justify your final decision with evidence—such as validation scores, visualizations, or model diagnostics.

3. Lessons Learned

- What insights did you gain about your dataset or your modeling process through this end-to-end workflow?

- If you had more time or data, what would you explore next?

> Your text here